# ReqMind AI — Model Optimization: Step 3
## Fine-Tuning `microsoft/deberta-v3-small` for FR / NFR Requirement Classification

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZAKARIAE48CHELLE/reqmind-ai/blob/main/notebooks/ReqMind_AI_Step3_DeBERTa_FineTuning.ipynb)

---

### Experiment Overview
This notebook implements **Step 3** of the ReqMind AI Model Optimization phase:
- **Base Architecture:** `microsoft/deberta-v3-small` (44M parameters, disentangled attention + enhanced masked language modeling).
- **Objective:** Binary classification into **Functional Requirements (FR)** and **Non-Functional Requirements (NFR)**, targeting nuanced usability, accessibility, and security boundaries.
- **Strict Partitioning & Zero Leakage:**
  - Audited Clean Population: **5,936 requirements** (from `Data/FR_NFR_Dataset.xlsx`, 50 contradictory records excluded).
  - Official Held-Out Test Set: **1,188 requirements** (20% sealed test set — **STRICTLY SEALED**).
  - Training Sub-Split: **3,798 requirements** (80% of training data).
  - Validation Sub-Split: **950 requirements** (20% of training data).
- **Model Selection & Regularization:** Checkpoints evaluated each epoch by **Validation Macro-F1** with early stopping (patience=2).
- **Benchmark Comparison:** Direct fair comparison on the exact 950 validation split against Baseline TF-IDF Linear SVM and Step 2 Calibrated Word+Char SVM.

## 1. Environment & Hardware Verification (GPU Check)
Ensure GPU acceleration is active in Colab (**Runtime > Change runtime type > T4 GPU**).

In [ ]:
# Check GPU availability and driver
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name:     {torch.cuda.get_device_name(0)}")
    print(f"Device VRAM:     {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("[!] WARNING: No GPU detected. Please go to Runtime -> Change runtime type -> T4 GPU.")

## 2. Install Required Dependencies
Install Hugging Face `transformers`, `datasets`, `accelerate`, and evaluation tools.

In [ ]:
!pip install -q transformers datasets accelerate evaluate scikit-learn matplotlib seaborn

## 3. Clone Repository or Load Audited Splits
We clone the official ReqMind AI repository to access the audited Step 3 splits directly.

In [ ]:
import os
from pathlib import Path

# Clone repository if not already in reqmind-ai directory
if not Path("Data/splits_step3").exists():
    !git clone https://github.com/ZAKARIAE48CHELLE/reqmind-ai.git
    %cd reqmind-ai

print("Working directory:", os.getcwd())
!ls -la Data/splits_step3/

## 4. Dataset Inspection & Integrity Verification

In [ ]:
import pandas as pd
import numpy as np

train_df = pd.read_csv("Data/splits_step3/deberta_train.csv")
val_df   = pd.read_csv("Data/splits_step3/deberta_val.csv")
test_df  = pd.read_csv("Data/splits_step3/deberta_test_sealed.csv")

print(f"[*] Training Sub-Split:   {len(train_df):,} rows (FR: {(train_df.label=='FR').sum():,}, NFR: {(train_df.label=='NFR').sum():,})")
print(f"[*] Validation Sub-Split: {len(val_df):,} rows   (FR: {(val_df.label=='FR').sum():,}, NFR: {(val_df.label=='NFR').sum():,})")
print(f"[*] Sealed Test Set:      {len(test_df):,} rows  (FR: {(test_df.label=='FR').sum():,}, NFR: {(test_df.label=='NFR').sum():,}) [SEALED]")

# Check for zero overlap
tr_keys = set(train_df.text.str.lower().str.strip())
val_keys = set(val_df.text.str.lower().str.strip())
test_keys = set(test_df.text.str.lower().str.strip())

assert len(tr_keys.intersection(val_keys)) == 0, "Data leakage between Train and Validation!"
assert len(tr_keys.intersection(test_keys)) == 0, "Data leakage between Train and Test!"
assert len(val_keys.intersection(test_keys)) == 0, "Data leakage between Validation and Test!"
print("[*] Data integrity check: PASSED. Zero exact text overlap across partitions.")

## 5. Tokenization with DeBERTa-v3-small Tokenizer
Load `AutoTokenizer` and format datasets for the Hugging Face `Trainer`.

In [ ]:
from transformers import AutoTokenizer, DataCollatorWithPadding
from datasets import Dataset

MODEL_ID = "microsoft/deberta-v3-small"
LABEL2ID = {"FR": 0, "NFR": 1}
ID2LABEL = {0: "FR", 1: "NFR"}

print(f"[*] Loading tokenizer: {MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

def tokenize_batch(batch):
    return tokenizer(batch["text"], truncation=True, max_length=96)

train_df["target"] = train_df["label"].map(LABEL2ID)
val_df["target"]   = val_df["label"].map(LABEL2ID)

train_ds = Dataset.from_pandas(train_df[["sample_id", "text", "target"]].rename(columns={"target": "label"}))
val_ds   = Dataset.from_pandas(val_df[["sample_id", "text", "target"]].rename(columns={"target": "label"}))

tokenized_train = train_ds.map(tokenize_batch, batched=True, remove_columns=["text", "sample_id"])
tokenized_val   = val_ds.map(tokenize_batch, batched=True, remove_columns=["text", "sample_id"])
data_collator   = DataCollatorWithPadding(tokenizer=tokenizer)

print(f"[*] Tokenization complete. Example input_ids length: {len(tokenized_train[0]['input_ids'])}")

## 6. Evaluation Metrics & Custom Logger Callback
Define `compute_metrics` to compute Macro-F1, Accuracy, and per-class metrics at each evaluation epoch.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
from transformers import TrainerCallback

class MetricHistoryCallback(TrainerCallback):
    def __init__(self):
        super().__init__()
        self.history = []

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics:
            self.history.append({
                "epoch": state.epoch,
                "step": state.global_step,
                "eval_loss": metrics.get("eval_loss"),
                "eval_accuracy": metrics.get("eval_accuracy"),
                "eval_macro_f1": metrics.get("eval_macro_f1"),
                "eval_f1_fr": metrics.get("eval_f1_fr"),
                "eval_f1_nfr": metrics.get("eval_f1_nfr"),
            })

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    macro_f1 = f1_score(labels, preds, average="macro")
    f1_fr = f1_score(labels, preds, pos_label=0)
    f1_nfr = f1_score(labels, preds, pos_label=1)
    prec_fr = precision_score(labels, preds, pos_label=0, zero_division=0)
    rec_fr = recall_score(labels, preds, pos_label=0, zero_division=0)
    prec_nfr = precision_score(labels, preds, pos_label=1, zero_division=0)
    rec_nfr = recall_score(labels, preds, pos_label=1, zero_division=0)
    return {
        "accuracy": round(float(acc), 4),
        "macro_f1": round(float(macro_f1), 4),
        "f1_fr": round(float(f1_fr), 4),
        "precision_fr": round(float(prec_fr), 4),
        "recall_fr": round(float(rec_fr), 4),
        "f1_nfr": round(float(f1_nfr), 4),
        "precision_nfr": round(float(prec_nfr), 4),
        "recall_nfr": round(float(rec_nfr), 4),
    }

## 7. Model Initialization & Fine-Tuning Setup
Configure `TrainingArguments` with:
- Learning Rate: $2 \times 10^{-5}$
- Warmup Ratio: $0.1$
- Weight Decay: $0.01$
- Early Stopping: Patience 2 on `eval_macro_f1`
- `load_best_model_at_end=True` to retain the optimal checkpoint.

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    num_labels=2,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

metric_logger = MetricHistoryCallback()
early_stopping = EarlyStoppingCallback(early_stopping_patience=2)

training_args = TrainingArguments(
    output_dir="checkpoints_deberta",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    load_best_model_at_end=True,
    fp16=torch.cuda.is_available(),
    logging_strategy="epoch",
    save_total_limit=2,
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[early_stopping, metric_logger],
)

print("[*] Trainer initialized and ready for execution.")

## 8. Execute DeBERTa-v3-small Fine-Tuning

In [ ]:
import time

print("[*] Commencing DeBERTa-v3-small training...")
t0 = time.perf_counter()
train_result = trainer.train()
duration = round(time.perf_counter() - t0, 2)
print(f"[*] Fine-tuning finished in {duration} seconds ({duration/60:.2f} minutes).")

## 9. Comprehensive Validation Evaluation & Metric Extraction

In [ ]:
# Evaluate best model
eval_results = trainer.evaluate()
print("=" * 60)
print("DEBERTA-V3-SMALL VALIDATION RESULTS (N=950)")
print("=" * 60)
print(f"Accuracy:  {eval_results['eval_accuracy']*100:.2f}%")
print(f"Macro-F1:  {eval_results['eval_macro_f1']*100:.2f}%")
print(f"FR-F1:     {eval_results['eval_f1_fr']*100:.2f}% (Precision: {eval_results['eval_precision_fr']*100:.2f}%, Recall: {eval_results['eval_recall_fr']*100:.2f}%)")
print(f"NFR-F1:    {eval_results['eval_f1_nfr']*100:.2f}% (Precision: {eval_results['eval_precision_nfr']*100:.2f}%, Recall: {eval_results['eval_recall_nfr']*100:.2f}%)")
print(f"Loss:      {eval_results['eval_loss']:.4f}")

## 10. Confusion Matrix & Training Dynamics Visualization

In [ ]:
import matplotlib.pyplot as plt

# Predictions on validation set
predictions = trainer.predict(tokenized_val)
logits = predictions.predictions
probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
pred_indices = np.argmax(probs, axis=1)
pred_labels = [ID2LABEL[i] for i in pred_indices]
confidences = np.max(probs, axis=1)

cm = confusion_matrix(val_df["label"], pred_labels, labels=["FR", "NFR"])

# 1. Plot Confusion Matrix
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, interpolation="nearest", cmap=plt.cm.Purples)
ax.figure.colorbar(im, ax=ax)
ax.set(
    xticks=[0, 1], yticks=[0, 1],
    xticklabels=["FR", "NFR"], yticklabels=["FR", "NFR"],
    ylabel="Ground Truth", xlabel="Predicted Label",
    title=f"DeBERTa-v3-small Validation Confusion Matrix\nAccuracy: {eval_results['eval_accuracy']*100:.2f}% | Macro-F1: {eval_results['eval_macro_f1']*100:.2f}%"
)
thresh = cm.max() / 2.0
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, format(cm[i, j], "d"),
                ha="center", va="center",
                color="white" if cm[i, j] > thresh else "black",
                fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("step3_deberta_val_confusion_matrix.png", dpi=200)
plt.show()

# 2. Plot Training and Validation Curves
if metric_logger.history:
    history_df = pd.DataFrame(metric_logger.history)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

    ax1.plot(history_df["epoch"], history_df["eval_loss"], marker="o", color="crimson", linewidth=2, label="Validation Loss")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.set_title("Validation Loss Trajectory")
    ax1.grid(True, linestyle=":", alpha=0.6)
    ax1.legend()

    ax2.plot(history_df["epoch"], history_df["eval_macro_f1"], marker="s", color="indigo", linewidth=2, label="Validation Macro-F1")
    ax2.plot(history_df["epoch"], history_df["eval_f1_nfr"], marker="^", color="darkorange", linestyle="--", label="Validation NFR-F1")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("F1 Score")
    ax2.set_title("Validation F1 Performance Trajectory")
    ax2.grid(True, linestyle=":", alpha=0.6)
    ax2.legend()

    plt.tight_layout()
    plt.savefig("step3_deberta_training_curves.png", dpi=200)
    plt.show()

## 11. Inference Latency Benchmark

In [ ]:
# Latency benchmark on 100 test samples
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.eval()
benchmark_texts = val_df["text"].iloc[:100].tolist()

with torch.no_grad():
    enc = tokenizer(benchmark_texts, padding=True, truncation=True, max_length=96, return_tensors="pt").to(device)
    # Warmup
    _ = model(**enc)
    t_start = time.perf_counter()
    _ = model(**enc)
    latency_ms = ((time.perf_counter() - t_start) / 100.0) * 1000.0

print(f"[*] Batched Inference Latency: {latency_ms:.3f} ms per requirement on {device}")

## 12. Fair Comparison Against Baseline & Optimized SVM
We compare the fine-tuned DeBERTa model against the Baseline SVM and Step 2 Optimized SVM on the exact same 950-sample validation split.

In [ ]:
# Recomputed exact validation performance of SVM baselines
comparison_table = pd.DataFrame([
    {
        "Model": "Baseline Linear SVM (Word TF-IDF)",
        "Representation": "Word n-grams (1,2)",
        "Accuracy": "87.47%",
        "Macro-F1": "85.90%",
        "FR-F1": "90.61%",
        "NFR-F1": "81.20%",
        "Latency": "~0.12 ms (CPU)",
    },
    {
        "Model": "Optimized SVM (Word+Char Balanced)",
        "Representation": "Word (1,2) + Char (3,5)",
        "Accuracy": "86.63%",
        "Macro-F1": "85.25%",
        "FR-F1": "89.77%",
        "NFR-F1": "80.73%",
        "Latency": "~0.15 ms (CPU)",
    },
    {
        "Model": "microsoft/deberta-v3-small (Fine-Tuned)",
        "Representation": "Disentangled Transformer (44M params)",
        "Accuracy": f"{eval_results['eval_accuracy']*100:.2f}%",
        "Macro-F1": f"{eval_results['eval_macro_f1']*100:.2f}%",
        "FR-F1": f"{eval_results['eval_f1_fr']*100:.2f}%",
        "NFR-F1": f"{eval_results['eval_f1_nfr']*100:.2f}%",
        "Latency": f"{latency_ms:.2f} ms (GPU)",
    },
])

import IPython.display as disp
disp.display(comparison_table)

## 13. Export Validation Errors Log
Save all misclassified samples to CSV for qualitative linguistic inspection.

In [ ]:
val_errors_df = val_df.copy()
val_errors_df["predicted_label"] = pred_labels
val_errors_df["confidence"] = np.round(confidences, 4)
val_errors_df["prob_FR"] = np.round(probs[:, 0], 4)
val_errors_df["prob_NFR"] = np.round(probs[:, 1], 4)
val_errors_df["is_correct"] = val_errors_df["label"] == val_errors_df["predicted_label"]

errors_only = val_errors_df[~val_errors_df["is_correct"]].reset_index(drop=True)
errors_only.to_csv("step3_deberta_val_errors.csv", index=False, encoding="utf-8")
print(f"[*] Exported {len(errors_only)} misclassifications to step3_deberta_val_errors.csv")
errors_only[["sample_id", "label", "predicted_label", "confidence", "text"]].head(10)

## 14. Save Model Artifacts & 1-Click Download Bundle
Saves the fine-tuned model and tokenizer, creates a zip bundle, and allows direct download back to the local repository.

In [ ]:
MODEL_EXPORT_DIR = "models/deberta_v3_small_fr_nfr"
os.makedirs(MODEL_EXPORT_DIR, exist_ok=True)

print(f"[*] Saving model and tokenizer to {MODEL_EXPORT_DIR}...")
trainer.save_model(MODEL_EXPORT_DIR)
tokenizer.save_pretrained(MODEL_EXPORT_DIR)

# Create zip bundle for 1-click download
!zip -r deberta_v3_small_reqmind.zip {MODEL_EXPORT_DIR} step3_deberta_val_confusion_matrix.png step3_deberta_training_curves.png step3_deberta_val_errors.csv

# In Google Colab, trigger download
try:
    from google.colab import files
    files.download("deberta_v3_small_reqmind.zip")
    print("[*] Download dialog initiated for deberta_v3_small_reqmind.zip")
except ImportError:
    print("[*] Running locally; zip bundle saved at deberta_v3_small_reqmind.zip")